# ZelusBench: Attention Updating

**Can the model update its representation after geometric transforms?**

Varies transform density while randomizing ALL background conditions
(chain depth, DAG structure, distractors, dimensionality, point types).

- Levels: Static (0), Light (2), Heavy (4), Extreme (7)
- 10 seeds per level = 40 scenarios, ~120 queries

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import ScenarioConfig, TransformDensity
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd))
            for qd, rp in zip(query_dicts, parts)]


TRANSFORM_LEVELS = [
    (0, TransformDensity.STATIC),
    (2, TransformDensity.LIGHT),
    (4, TransformDensity.HEAVY),
    (7, TransformDensity.EXTREME),
]
SEEDS = 10
TOTAL = len(TRANSFORM_LEVELS) * SEEDS

print(f"ZelusBench Attention Updating")
print(f"Transforms: {[t for t, _ in TRANSFORM_LEVELS]} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_attention_updating")
def zelusbench_attention_updating(llm) -> tuple[float, float]:
    """Attention updating: accuracy vs transform count (randomized background).
    Returns (mean_accuracy, std_dev)."""

    _start = time.time()
    print(f"Running {TOTAL} scenarios across {len(TRANSFORM_LEVELS)} transform levels...")
    print("=" * 60)

    all_scores = []
    level_scores = {}
    scenario_num = 0

    for num_t, density in TRANSFORM_LEVELS:
        level_scores[num_t] = []
        for seed in range(SEEDS):
            scenario_num += 1
            unique_seed = num_t * 1000 + seed
            rng = random.Random(unique_seed)

            # Scale transform types with density
            if density == TransformDensity.STATIC:
                tt = []
            elif density == TransformDensity.LIGHT:
                tt = ["rotation", "translation"]
            else:
                tt = ["rotation", "translation", "reflection", "scaling"]

            cfg = ScenarioConfig.randomize_except(rng, pinned={
                "transform_density": density,
                "transform_types": tt,
                "num_queries": 3,
                "seed": unique_seed,
            })
            gen = ScenarioGenerator(cfg)
            scenario = gen.generate(f"updating_t{num_t}_s{seed}")

            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            all_scores.extend(scores)

            avg = float(np.mean([s.score for s in scores]))
            level_scores[num_t].append(avg)

            q_depths = [s.chain_depth for s in scores]
            tiers = [s.tier.name for s in scores]
            bg = f"dim={cfg.dim} dag={cfg.dag_structure.name} depth={cfg.min_chain_depth}-{cfg.max_chain_depth} dist={cfg.distractor_level.name}"
            print(f"  [{scenario_num}/{TOTAL}] transforms={num_t} seed={seed} "
                  f"avg={avg:.2f} q_depths={q_depths} tiers={tiers} | {bg}")

        level_avg = float(np.mean(level_scores[num_t]))
        print(f"  >> {num_t} transforms mean: {level_avg:.3f}")

    print("\n" + "=" * 60)
    print("RESULTS BY TRANSFORM COUNT:")
    static_acc = float(np.mean(level_scores.get(0, [0])))
    for num_t, _ in TRANSFORM_LEVELS:
        avg = float(np.mean(level_scores[num_t]))
        drop = static_acc - avg if static_acc > 0 else 0
        print(f"  transforms={num_t}  accuracy={avg:.3f}  drop_from_static={drop:+.3f}")
        kbench.assertions.assert_true(
            avg >= 0,
            expectation=f"{num_t} transforms: model should produce valid answers (got {avg:.3f})"
        )

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start

    print(f"\nOverall: {overall:.3f} +/- {std:.3f}")
    print(f"Total queries: {len(all_scores)}")
    print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    return overall, std


zelusbench_attention_updating.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attention_updating